In [1]:
import os
import torch
from torch import nn
import torch.optim as optim
import time
from tqdm import tqdm
import datasets
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from peft import LoraConfig, get_peft_model
from transformers import BitsAndBytesConfig
import open_clip
import torch.nn.utils.rnn as rnn_utils
from argparse import ArgumentParser
from peft import PeftModel
import torchvision.transforms.functional as TF

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


class CLIP_Adapted_ImageEncoder(nn.Module):
    def __init__(self, clip_model):
        super().__init__()
        self.clip_model = clip_model

        self.adapter = nn.Sequential(
            nn.Linear(512, 512),
            nn.LayerNorm(512),

            nn.LeakyReLU(),

            nn.Linear(512, 512),
            nn.LayerNorm(512),

            nn.LeakyReLU(),

            nn.Linear(512, 512),
            nn.LayerNorm(512)
        )

    def forward(self, image):
        return self.adapter(self.clip_model.encode_image(image))

/home/mmuuser/anaconda3/envs/vlunet/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


/home/mmuuser/anaconda3/envs/vlunet/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
class CLIP_Adapted_TextEncoder(nn.Module):
    def __init__(self, clip_model):
        super().__init__()
        self.clip_model = clip_model
        self.adapter = nn.Sequential(
            nn.Linear(512, 512),
            nn.LayerNorm(512),

            nn.LeakyReLU(),

            nn.Linear(512, 512),
            nn.LayerNorm(512),

            nn.LeakyReLU(),

            nn.Linear(512, 512),
            nn.LayerNorm(512)
        )

    def forward(self, text):
        return self.adapter(self.clip_model.encode_text(text))

class DA_adapter(nn.Module):
    def __init__(self, clip_model):
        super().__init__()
        for param in clip_model.parameters():
            param.requires_grad = False
        self.ad_imageEncoder = CLIP_Adapted_ImageEncoder(clip_model=clip_model)
        self.ad_textEncoder = CLIP_Adapted_TextEncoder(clip_model=clip_model)

    def forward(self, image, text):
        image = self.ad_imageEncoder(image)
        text = self.ad_textEncoder(text)
        return image, text

In [3]:
parser = ArgumentParser(description='LDUN')
parser.add_argument('--about', type=str, default='5task')
parser.add_argument('--start_epoch', type=int, default=0, help='epoch number of start training')
parser.add_argument('--end_epoch', type=int, default=100, help='epoch number of end training')
parser.add_argument('--learning_rate', type=float, default=1e-5, help='learning rate')
parser.add_argument('--resume', type=bool, default=False, help='is resume')
parser.add_argument('--group_num', type=int, default=1, help='group number for training')
parser.add_argument('--gpu_list', type=str, default='1', help='gpu index')
parser.add_argument('--checkpoints_dir', type=str, default='ckpt/Phase3/BLIP_CLIP_degradation_extractor/Checkpoint', help='checkpoints dir')
parser.add_argument('--log_dir', type=str, default='log', help='log directory')
parser.add_argument('--ext', type=str, default='.png', help='training data directory')
parser.add_argument('--is_aug', type=bool,default=False, help='is aug')
parser.add_argument('--is_clip_tuning', type=bool,default=False, help='is finetuning clip')
parser.add_argument('--patch_size', type=int, default=128, help='patchsize of input.')

# Noise, Haze, Rain, Blurr, Lowlight
NHRBL = ["./datasets/denoising_datasets/15_train_paths.txt",
 "./datasets/denoising_datasets/25_train_paths.txt",
 "./datasets/denoising_datasets/50_train_paths.txt",
 "./datasets/dehazing_datasets/train_paths.txt",
 "./datasets/deraining_datasets/Rain100L/train_paths.txt",
 "./datasets/deblurring_datasets/GoPro/train_paths.txt",
 "./datasets/delowlight_datasets/LoL/train_paths.txt"]

In [4]:
args = parser.parse_args(args=[])
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = args.gpu_list
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda:0


In [5]:
# !pip install --upgrade torchao

blip2_model_path = "ckpt/Phase2/Blip2DegradationCaption/best_finetuned_model"
print("Loading BLIP‑2 processor and model...")

model_name = "Salesforce/blip2-opt-2.7b"

processor = Blip2Processor.from_pretrained(model_name)
processor.tokenizer.pad_token = processor.tokenizer.eos_token

# bnb_config = BitsAndBytesConfig(load_in_8bit=True)

base_model = Blip2ForConditionalGeneration.from_pretrained(
    model_name,
    # quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto"
)

model_inf = PeftModel.from_pretrained(base_model, blip2_model_path)
model_inf.eval()
model_inf.to(device)
torch.set_grad_enabled(False)

Loading BLIP‑2 processor and model...


Loading weights: 100%|██████████| 1247/1247 [00:00<00:00, 2576.11it/s]


torch.autograd.grad_mode.set_grad_enabled(mode=False)

In [6]:
model, preprocess, _ = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
tokenizer = open_clip.get_tokenizer('ViT-B-32')

model = DA_adapter(model)
model.to(device)
model.eval()

DA_adapter(
  (ad_imageEncoder): CLIP_Adapted_ImageEncoder(
    (clip_model): CLIP(
      (visual): VisionTransformer(
        (patchnorm_pre_ln): Identity()
        (conv1): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
        (patch_dropout): Identity()
        (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (transformer): Transformer(
          (resblocks): ModuleList(
            (0-11): 12 x ResidualAttentionBlock(
              (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
              (attn): MultiheadAttention(
                (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
              )
              (ls_1): Identity()
              (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
              (mlp): Sequential(
                (c_fc): Linear(in_features=768, out_features=3072, bias=True)
                (gelu): GELU(approxima

In [7]:
@torch.inference_mode()
def generate_caption_batch(image_paths, model, processor, device):

    images = [
        Image.open(p).convert("RGB")
        for p in image_paths
    ]

    prompt = "Question: Describe the scene. Answer:"

    inputs = processor(
        images=images,
        text=[prompt] * len(images),
        return_tensors="pt",
        padding=True
    ).to(device)

    generated_ids = model.generate(
            **inputs,
            max_new_tokens=60,
            num_beams=1,
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,
            length_penalty=0.8,
            early_stopping=True,
            eos_token_id=processor.tokenizer.eos_token_id,
            pad_token_id=processor.tokenizer.pad_token_id
        )

    captions = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )

    captions = [
        c.replace(prompt, "").strip()
        for c in captions
    ]

    return captions

In [8]:
BASE_DIR = "datasets"

def fix_path(p):
    return os.path.join(BASE_DIR, p.replace("./", "").strip())

all_image_paths = []

print("🔍 Building dataset paths...")

for file in tqdm(NHRBL, desc="📂 Dataset Files"):

    with open(file, "r") as f:
        lines = [l.strip() for l in f if l.strip()]

    for line in lines:

        if "," not in line:
            continue

        noisy_path, _ = line.split(",")

        noisy_path = fix_path(noisy_path)

        if os.path.exists(noisy_path):
            all_image_paths.append(noisy_path)

print("\n✅ TOTAL IMAGES:", len(all_image_paths))

🔍 Building dataset paths...


📂 Dataset Files: 100%|██████████| 7/7 [00:00<00:00, 29.28it/s]


✅ TOTAL IMAGES: 90752


In [9]:
BASE_SAVE = "ckpt/Phase3/BLIP_CLIP_degradation_extractor/Captions"

os.makedirs(BASE_SAVE, exist_ok=True)

CSV_PATH   = os.path.join(BASE_SAVE, "captions_dataset.csv")
CACHE_PATH = os.path.join(BASE_SAVE, "captions_cache.pt")
STATE_PATH = os.path.join(BASE_SAVE, "captions_state.pt")

SAVE_EVERY = 1000
BATCH_SIZE = 32

In [10]:
class DatasetManager:
    def __init__(self, csv_path, cache_path, state_path):

        self.csv_path = csv_path
        self.cache_path = cache_path
        self.state_path = state_path

        self.df = pd.DataFrame(columns=["image_path", "caption"])
        self.cache = {}
        self.state = {"processed": set()}

        self._load_all()

    def _load_all(self):

        # CSV
        if os.path.exists(self.csv_path):
            self.df = pd.read_csv(self.csv_path)
            print(f"[CSV] Loaded {len(self.df)} rows")

        # CACHE
        if os.path.exists(self.cache_path):
            self.cache = torch.load(self.cache_path)
            print(f"[CACHE] Loaded {len(self.cache)} items")

        # STATE
        if os.path.exists(self.state_path):
            self.state = torch.load(self.state_path)
            print(f"[STATE] Resumed {len(self.state['processed'])} items")

        # sync CSV → cache
        for _, row in self.df.iterrows():
            self.cache[row["image_path"]] = row["caption"]

    def add_batch(self, batch_data):
        """
        batch_data: list of (path, caption)
        """
        df_new = pd.DataFrame(batch_data, columns=["image_path", "caption"])
        self.df = pd.concat([self.df, df_new], ignore_index=True)

        for p, c in batch_data:
            self.cache[p] = c
            self.state["processed"].add(p)

    def exists(self, path):
        return path in self.cache or path in self.state["processed"]

    def save(self):
        self.df.to_csv(self.csv_path, index=False)
        torch.save(self.cache, self.cache_path)
        torch.save(self.state, self.state_path)

manager = DatasetManager(CSV_PATH, CACHE_PATH, STATE_PATH)

[CSV] Loaded 90752 rows
[CACHE] Loaded 90752 items
[STATE] Resumed 90752 items


In [11]:
remaining_paths = [p for p in all_image_paths if not manager.exists(p)]

print("TOTAL:", len(all_image_paths))
print("REMAINING:", len(remaining_paths))

buffer = []

for i in tqdm(range(0, len(remaining_paths), BATCH_SIZE), desc="Captioning"):

    batch_paths = remaining_paths[i:i+BATCH_SIZE]

    captions = generate_caption_batch(
        batch_paths,
        model_inf,
        processor,
        device
    )

    buffer.extend(list(zip(batch_paths, captions)))

    if len(buffer) >= SAVE_EVERY:
        manager.add_batch(buffer)
        buffer.clear()
        manager.save()

        print(f"Saved {len(manager.df)} captions")

TOTAL: 90752
REMAINING: 0


Captioning: 0it [00:00, ?it/s]


In [12]:
print("ALL IMAGES:", len(all_image_paths))
print("ALREADY DONE:", len(manager.state["processed"]))
print("TO PROCESS:", len([p for p in all_image_paths if not manager.exists(p)]))

ALL IMAGES: 90752
ALREADY DONE: 90752
TO PROCESS: 0


In [13]:
if len(buffer) > 0:
    manager.add_batch(buffer)
    manager.save()

print("\nDONE")
print("Total captions:", len(manager.df))


DONE
Total captions: 90752


In [14]:
clip_model, preprocess, _ = open_clip.create_model_and_transforms(
    'ViT-B-32',
    pretrained='laion2b_s34b_b79k'
)

model = DA_adapter(clip_model).to(device)
model.train()


for name, param in model.named_parameters():
    if "adapter" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

trainable = sum(p.requires_grad for p in model.parameters())
print("Trainable params:", trainable)

Trainable params: 24


In [15]:
class FastDataset(Dataset):
    def __init__(self, image_paths, captions_cache, preprocess):
        self.image_paths = image_paths
        self.captions_cache = captions_cache
        self.preprocess = preprocess

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]

        image = Image.open(path).convert("RGB")
        image = self.preprocess(image)

        caption = self.captions_cache[path]

        return image, caption

In [16]:
captions_cache = torch.load(CACHE_PATH, map_location="cpu")
print("Loaded captions:", len(captions_cache))

train_image_paths = [
    p for p in all_image_paths
    if p in captions_cache
]

print("Train images:", len(train_image_paths))

Loaded captions: 90752
Train images: 90752


In [17]:
dataset = FastDataset(
    train_image_paths,
    captions_cache,
    preprocess
)

train_loader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True,
    num_workers=12,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4
)

val_loader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True,
    num_workers=12,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4
)


In [18]:
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=args.learning_rate,
    betas=(0.9, 0.98),
    weight_decay=0.01
)

criterion = torch.nn.CrossEntropyLoss()
criterion = nn.CrossEntropyLoss()

temperature = 0.07

In [19]:
# =====================================================
# TEST ONE CAPTION + TOKENIZATION
# =====================================================

print("Caption cache loaded:", len(captions_cache))

sample_path = all_image_paths[0]

print("=" * 80)
print("SAMPLE IMAGE PATH:")
print(sample_path)

print("\nBLIP GENERATED CAPTION:")
print(captions_cache[sample_path])

tokens = tokenizer([captions_cache[sample_path]])

print("\nTOKEN TENSOR SHAPE:")
print(tokens.shape)

print("\nTOKEN IDS:")
print(tokens)

print("\nTOKEN IDS (LIST FORMAT):")
print(tokens[0].tolist())

print("=" * 80)

Caption cache loaded: 90752
SAMPLE IMAGE PATH:
datasets/denoising_datasets/BSD400/noisy15/test_001.png

BLIP GENERATED CAPTION:
This is a bear family in heavy noise conditions.

TOKEN TENSOR SHAPE:
torch.Size([1, 77])

TOKEN IDS:
tensor([[49406,   589,   533,   320,  4298,  1315,   530,  4200,  9307,  5892,
           269, 49407,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0]])

TOKEN IDS (LIST FORMAT):
[49406, 589, 533, 320, 4298, 1315, 530, 4200, 9307, 5892, 269, 49407, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [20]:
start_epoch = args.start_epoch

DRIVE_SAVE_DIR = args.checkpoints_dir

os.makedirs(
    DRIVE_SAVE_DIR,
    exist_ok=True
)

latest_ckpt_path = os.path.join(
    DRIVE_SAVE_DIR,
    "model_latest.pth"
)
best_ckpt_path = os.path.join(
    DRIVE_SAVE_DIR,
    "model_best.pth"
)

def load_checkpoint(ckpt_path):
    print(f"[INFO] Loading checkpoint: {ckpt_path}")
    checkpoint = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])

    if "optimizer_state_dict" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    elif "optimizer" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer"])
    else:
        print("[WARNING] Optimizer state not found in checkpoint.")

    return checkpoint["epoch"] + 1

if os.path.exists(latest_ckpt_path):
    try:
        start_epoch = load_checkpoint(latest_ckpt_path)
        print(f"[INFO] Resume Epoch {start_epoch} from latest")
    except Exception as e:
        print(f"[ERROR] Failed to load latest checkpoint: {e}")
        if os.path.exists(best_ckpt_path):
            try:
                print("[INFO] Attempting to load best checkpoint instead...")
                start_epoch = load_checkpoint(best_ckpt_path)
                print(f"[INFO] Resume Epoch {start_epoch} from best")
            except Exception as e2:
                print(f"[ERROR] Failed to load best checkpoint as well: {e2}")
                print("[INFO] Training From Scratch")
        else:
            print("[INFO] Training From Scratch")
else:
    print("[INFO] Training From Scratch")


[INFO] Loading checkpoint: ckpt/Phase3/BLIP_CLIP_degradation_extractor/Checkpoint/model_latest.pth
[INFO] Resume Epoch 100 from latest


In [21]:
try:
    # Re-enable gradient tracking that was turned off during BLIP-2 loading
    torch.set_grad_enabled(True)

    # Get one batch from the training loader to define image_features and text_features
    images, captions = next(iter(train_loader))

    images = images.to(device, non_blocking=True)
    texts = tokenizer(list(captions)).to(device)

    # Ensure model is in training mode to track gradients
    model.train()

    # Perform a forward pass
    image_features, text_features = model(images, texts)

    # Calculate a dummy loss to define the 'loss' variable
    image_features_norm = image_features / image_features.norm(dim=-1, keepdim=True)
    text_features_norm = text_features / text_features.norm(dim=-1, keepdim=True)
    logits = (image_features_norm @ text_features_norm.T) / temperature
    targets = torch.arange(len(images), device=device)
    loss = criterion(logits, targets)

    print("image requires_grad:", image_features.requires_grad)
    print("text requires_grad:", text_features.requires_grad)
    print("loss requires_grad:", loss.requires_grad)
except NameError as e:
    print(f"Error: {e}. Please ensure `train_loader`, `tokenizer`, `model`, `device`, `temperature`, and `criterion` are defined and accessible.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

image requires_grad: True
text requires_grad: True
loss requires_grad: True


In [22]:
# Re-enable gradient tracking
torch.set_grad_enabled(True)

train_losses = []
train_accs = []

val_losses = []
val_accs = []

best_val_acc = 0.0

for epoch in range(start_epoch, args.end_epoch):

    # TRAINING
    model.train()

    epoch_loss = 0.0
    epoch_correct = 0
    epoch_total = 0

    pbar = tqdm(train_loader, desc=f"Train Epoch {epoch}")

    for images, captions in pbar:

        images = images.to(device, non_blocking=True)
        texts = tokenizer(list(captions)).to(device)

        # Forward
        image_features, text_features = model(images, texts)

        image_features = image_features / image_features.norm(
            dim=-1, keepdim=True
        )
        text_features = text_features / text_features.norm(
            dim=-1, keepdim=True
        )

        logits = 100.0 * (image_features @ text_features.T)

        targets = torch.arange(len(images), device=device)

        loss = criterion(logits, targets)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        preds = logits.argmax(dim=1)
        batch_correct = (preds == targets).sum().item()

        epoch_correct += batch_correct
        epoch_total += len(images)

        pbar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "acc": f"{100 * batch_correct / len(images):.1f}%"
        })

    train_loss = epoch_loss / len(train_loader)
    train_acc = epoch_correct / epoch_total

    train_losses.append(train_loss)
    train_accs.append(train_acc)

    # VALIDATION

    model.eval()

    val_epoch_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        pbar_val = tqdm(val_loader, desc=f"Val Epoch {epoch}")

        for images, captions in pbar_val:

            images = images.to(device, non_blocking=True)
            texts = tokenizer(list(captions)).to(device)

            image_features, text_features = model(images, texts)

            image_features = image_features / image_features.norm(
                dim=-1, keepdim=True
            )
            text_features = text_features / text_features.norm(
                dim=-1, keepdim=True
            )

            logits = 100.0 * (image_features @ text_features.T)

            targets = torch.arange(len(images), device=device)

            loss = criterion(logits, targets)

            val_epoch_loss += loss.item()

            preds = logits.argmax(dim=1)

            val_correct += (preds == targets).sum().item()
            val_total += len(images)

            pbar_val.set_postfix({
                "loss": f"{loss.item():.4f}",
                "acc": f"{100 * (preds == targets).sum().item() / len(images):.1f}%"
            })

    val_loss = val_epoch_loss / len(val_loader)
    val_acc = val_correct / val_total

    val_losses.append(val_loss)
    val_accs.append(val_acc)

    # PRINT METRICS
    print(
        f"\nEpoch {epoch} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc : {train_acc*100:.2f}% | "
        f"Val Loss  : {val_loss:.4f} | "
        f"Val Acc   : {val_acc*100:.2f}% | "
    )

    learnable_params = {
        name: param.detach().cpu()
        for name, param in model.named_parameters()
        if param.requires_grad
    }

    # SAVE LATEST CHECKPOINT
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "learnable_params": learnable_params,
        "train_losses": train_losses,
        "train_accs": train_accs,
        "val_losses": val_losses,
        "val_accs": val_accs,
        "best_val_acc": best_val_acc,
    }

    torch.save(checkpoint, latest_ckpt_path)

    # SAVE BEST CHECKPOINT
    if val_acc > best_val_acc:

        best_val_acc = val_acc

        checkpoint["best_val_acc"] = best_val_acc

        torch.save(checkpoint, best_ckpt_path)

        print(
            f"[BEST] New best model saved "
            f"(Val Acc = {best_val_acc*100:.2f}%)"
        )

    print(
        f"[LATEST] Checkpoint saved "
        f"(Epoch {epoch})"
    )

In [2]:
!mkdir -p blip_vlunet_pretrained_ckpt
!cp -r ckpt/Phase3/BLIP_CLIP_degradation_extractor/Checkpoint/model_best.pth \
    blip_vlunet_pretrained_ckpt/blip_vlunet_clip_tuned.pth